# Step 0 - 파노라마 전처리 (4방향 크롭: 좌/정면/우/후방)

**목표**: 360° 파노라마(256×1024)를 수직으로 **4등분**하여 네 방향(좌·정면·우·후)을 모두 추출한다.
각 구간은 360° 중 90° 범위이므로 HFOV 90°로 핀홀 K를 계산한다(03 BEV에서 사용).

**방향 매핑**: idx0=left(yaw −90°), idx1=front(0°), idx2=right(+90°), idx3=back(180°).

**입력**: `test_img/*.jpg` (360° 파노라마, `exclude/`는 비재귀 glob으로 자연 제외)

**출력**: `test_output/00_front/{stem}_{dir}.jpg` + `{stem}_{dir}_cam.json` (K, hfov, direction, yaw_deg)

## 1. 라이브러리 및 파라미터

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

# 입출력 경로 ──────────────────────────────
IMG_DIR = Path("test_img")              # 360° 파노라마 (exclude/ 하위는 비재귀 glob으로 제외됨)
OUT_DIR = Path("test_output/00_front")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 4방향 크롭 파라미터 ────────────────────────
# 파노라마를 수평 4등분: (좌, 정면, 우, 후) 각 90°. yaw는 정면=0, 우측=+90, 좌측=−90, 후방=180.
N_SPLIT = 4
PANO_HFOV = 360.0
FRONT_HFOV = PANO_HFOV / N_SPLIT          # = 90°
# (idx, dir_name, yaw_deg)
DIRS = [(0, "left", -90.0), (1, "front", 0.0), (2, "right", 90.0), (3, "back", 180.0)]

IMG_PATHS = sorted(IMG_DIR.glob("*.jpg"))  # 비재귀: exclude/ 제외
print(f"파노라마 {len(IMG_PATHS)}장 발견 (exclude 제외)")

파노라마 573장 발견 (exclude 제외)


c:\Users\Jihyun\Desktop\git\ITS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 4등분 크롭 함수

파노라마를 수평 4등분하여 각 구간 `[i*seg_w : (i+1)*seg_w]`을 잘라 방향별 뷰를 생성.
각 구간은 90° 범위이므로 핀홀 intrinsics를 계산: `fx=(W/2)/tan(HFOV/2)`, `cx=W/2`, `cy=H/2`.

In [2]:
def make_crop(pano_bgr, idx, n_split=N_SPLIT, front_hfov=FRONT_HFOV):
    """파노라마를 수평 n등분 후 idx 구간 크롭. (크롭 이미지, 핀홀 K) 반환."""
    ph, pw = pano_bgr.shape[:2]
    seg_w = pw // n_split
    x0 = idx * seg_w
    x1 = x0 + seg_w
    crop = pano_bgr[:, x0:x1].copy()
    fh, fw = crop.shape[:2]

    fx = (fw / 2.0) / np.tan(np.radians(front_hfov) / 2.0)
    cx, cy = fw / 2.0, fh / 2.0
    K = np.array([[fx, 0, cx], [0, fx, cy], [0, 0, 1]], dtype=np.float64)
    return crop, K


print("크롭 함수 정의 완료")

크롭 함수 정의 완료


## 3. 전체 이미지 처리 및 저장 (파노라마당 4방향)

In [3]:
n_saved = 0
for img_path in tqdm(IMG_PATHS, desc="전처리(4방향)"):
    stem = img_path.stem
    pano = cv2.imread(str(img_path))
    if pano is None:
        continue
    ph, pw = pano.shape[:2]

    for idx, dir_name, yaw in DIRS:
        crop, K = make_crop(pano, idx)
        fh, fw = crop.shape[:2]

        cv2.imwrite(str(OUT_DIR / f"{stem}_{dir_name}.jpg"), crop)

        cam = {
            "src_image": img_path.name,
            "src_hw": [ph, pw],
            "out_hw": [fh, fw],
            "hfov_deg": FRONT_HFOV,
            "direction": dir_name,
            "yaw_deg": yaw,
            "split": [N_SPLIT, idx],
            "K": K.tolist(),
            "fx": K[0, 0], "fy": K[1, 1], "cx": K[0, 2], "cy": K[1, 2],
        }
        with open(OUT_DIR / f"{stem}_{dir_name}_cam.json", "w", encoding="utf-8") as f:
            json.dump(cam, f, ensure_ascii=False, indent=2)
        n_saved += 1

print(f"=== 전처리 완료: 파노라마 {len(IMG_PATHS)}장 × {len(DIRS)}방향 = {n_saved}장 저장 → {OUT_DIR} ===")
print("다음 단계: 01_segmentation.ipynb")

전처리(4방향): 100%|██████████| 573/573 [00:03<00:00, 181.81it/s]

=== 전처리 완료: 파노라마 573장 × 4방향 = 2292장 저장 → test_output\00_front ===
다음 단계: 01_segmentation.ipynb
